In [ ]:
import sys #有新檔案取代 --> vote/major_voted_preprocess
import pandas as pd
import os
import numpy as np
from sklearn.metrics import confusion_matrix

In [12]:
def load_file_name_non_file_ext(path, dataset, iteration, win_num_df = None):
    name = [] #原本只有一個名字的 MACE001，就會在 window_name 列表中變成 20 個連續的 MACE001
    for i in range(iteration):
        # if i == 1:
        #     break
        file_name = pd.read_csv(path + f"{dataset}_file_name_{i + 1}.csv")
        file_name = file_name.iloc[: , 0].tolist()
        name += file_name
    window_name = []
    if win_num_df is not None:
        file_name = win_num_df['file_name']
        for i, fname in enumerate(name):
            mask = file_name.isin([fname])
            win_num = win_num_df[mask]['window_per_person'].reset_index(drop=True)
            win_num = int(win_num.iloc[0])
            window_name += [name[i]] * win_num
    else:
        window_name = []
        for i in range(len(name)):
            window_name += [name[i]] * 141 # 如果切視窗，每個檔案對應 141 個視窗

    return window_name


def load_label(path, dataset, iteration):
    df = pd.DataFrame()
    for i in range(iteration):
        data = pd.read_csv(path + f"y_{dataset}_label_{i + 1}.csv")
        df = pd.concat([df, data], axis=0, ignore_index=True) #0:上下堆疊 1:左右

    return df


def load_prob(path, dataset, iteration):
    df = pd.DataFrame()
    for i in range(iteration):
        data = pd.read_csv(path + f"yhat_{dataset}_prob_{i + 1}.csv")
        df = pd.concat([df, data], axis=0, ignore_index=True)

    return df


def id_sort(data):
    data['SortKey'] = data.iloc[:, 0].str.extract(r'(\d+)').astype(int)  # 提取數字部分，轉換為數值型
    data_sorted = data.sort_values(by='SortKey').reset_index(drop=True)  # 按數字部分排序
    data_sorted = data_sorted.drop(columns=['SortKey'])  # 刪除輔助列

    return data_sorted


def calculate_sample_num(data_df_sorted): #驗算 若視窗不同 代表前面的資料處理出錯了（例如少讀了檔案，或重複讀了檔案）
    sample_counts = data_df_sorted.iloc[:, 0].value_counts().reset_index()
    # 重新命名欄位名稱
    sample_counts.columns = ['ID', 'Sample_num']
    # 按照 ID 排序（確保 MACE001、MACE002 順序正確）
    sample_counts['SortKey'] = sample_counts['ID'].str.extract(r'(\d+)').astype(int)
    sample_counts = sample_counts.sort_values(by='SortKey').drop(columns=['SortKey']).reset_index(drop=True)

    return sample_counts

In [13]:
file_path = r"D:\M143020071\multirocket_result\multirocket\resample_0\MultiRocket_50000\sr10000_500_2000_20s_logistic_longer_51_41_100pts_20win/"
per_person_win_num_dir = r"D:\M143020071\raw_data_result\SKNA_signal\ch1\sr10000_500_2000_20s_ECG_signal_rpeak_5_10min_longer_100pts_mr_20win/"
save_path = file_path + "combine/"

if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Directory {save_path} created.")
else:
    print(f"Directory {save_path} already exists.")

iter_round = 90

Directory D:\M143020071\multirocket_result\multirocket\resample_0\MultiRocket_50000\sr10000_500_2000_20s_logistic_longer_51_41_100pts_20win/combine/ created.


In [14]:
h_win_num = pd.read_csv(per_person_win_num_dir + "h_window_per_person.csv")
p_win_num = pd.read_csv(per_person_win_num_dir + "p_window_per_person.csv")
all_person_win_num = pd.concat([h_win_num, p_win_num], axis=0, ignore_index=True)
all_person_win_num = all_person_win_num.sort_values(by='file_name').reset_index(drop=True)
print(f"Healthy 人數: {len(h_win_num)}, Patient 人數: {len(p_win_num)}")
print(f"總人數: {len(all_person_win_num)}")
print(all_person_win_num.head()) # 印出前幾筆確認

Healthy 人數: 432, Patient 人數: 90
總人數: 522
  file_name  window_per_person
0   MACE001                 20
1   MACE002                 20
2   MACE003                 20
3   MACE004                 20
4   MACE005                 20


In [ ]:
train_file_name= load_file_name_non_file_ext(file_path, "train", iter_round, win_num_df = all_person_win_num)
test_file_name = load_file_name_non_file_ext(file_path, "test", iter_round, win_num_df = all_person_win_num)
#把「病人檔名」依照視窗數量複製展開（例如 MACE001 變 20 個），以跟預測數據對齊。

train_label = load_label(file_path, "train", iter_round)
test_label = load_label(file_path, "test", iter_round)
#把 90 輪產出的**「真實答案卷 (0/1)」**全部讀進來，黏成一張大總表。

train_prob = load_prob(file_path, "train", iter_round)
test_prob = load_prob(file_path, "test", iter_round)
#把 90 輪產出的**「預測分數 (機率)」**全部讀進來，黏成一張大總表。

train_file_name_df = pd.DataFrame(train_file_name, columns=["file_name"])
test_file_name_df = pd.DataFrame(test_file_name, columns=["file_name"])


train_label = pd.concat([train_file_name_df, train_label], axis=1)  #1:左右疊加
test_label = pd.concat([test_file_name_df, test_label], axis=1)


train_prob = pd.concat([train_file_name_df, train_prob], axis=1)
test_prob = pd.concat([test_file_name_df, test_prob], axis=1)


train_label_sorted = id_sort(train_label)
test_label_sorted = id_sort(test_label)
train_prob_sorted = id_sort(train_prob)
test_prob_sorted = id_sort(test_prob)
#依照 ID 的數字大小正確排序（避免 MACE10 排在 MACE2 前面）。

sample_num_train = calculate_sample_num(train_label_sorted)
sample_num_test = calculate_sample_num(test_label_sorted)
#驗算點名每個人有幾筆資料，用來檢查前面有沒有漏讀或讀錯

train_label_sorted.to_csv(save_path + "train_label.csv", index=False)
test_label_sorted.to_csv(save_path + "test_label.csv", index=False)
train_prob_sorted.to_csv(save_path + "train_prob.csv", index=False)
test_prob_sorted.to_csv(save_path + "test_prob.csv", index=False)
sample_num_train.to_csv(save_path + "sample_num_train.csv", index=False)
sample_num_test.to_csv(save_path + "sample_num_test.csv", index=False)

print(f"save path: {save_path}")

save path: D:\M143020071\multirocket_result\multirocket\resample_0\MultiRocket_50000\sr10000_500_2000_20s_logistic_longer_51_41_100pts_20win/combine/
